# Process Tree Anomaly Baseline

**Goal:** establish a parent-child process baseline across the estate and surface rare or unseen parent/child pairs for hunt-style review.

**Hypothesis link:** supports H-006 (rundll32 abuse), H-007 (process hollowing), and the broader "unusual ancestry" family of hunts.

**Inputs:**
- Elastic index pattern: `logs-endpoint.events.process-*`
- Lookback: 30 days

**Outputs:**
- A frequency table of `(process.parent.name, process.name)` pairs
- A ranked list of pairs seen < 0.01% of the time
- A scratch cell for pivoting into the raw events for any chosen pair

**Author:** v3nomtech  
**Last run:** 2026-05-12

## 1. Setup

In [ ]:
import os
import pandas as pd
from elasticsearch import Elasticsearch

ES_URL   = os.environ.get('ES_URL',   'https://elastic.local:9200')
ES_USER  = os.environ.get('ES_USER',  'hunter')
ES_PASS  = os.environ.get('ES_PASS')
INDEX    = 'logs-endpoint.events.process-*'
LOOKBACK = '30d'

es = Elasticsearch(ES_URL, basic_auth=(ES_USER, ES_PASS), verify_certs=False)
es.info()

## 2. Pull parent/child pairs

We aggregate on the server side — pulling raw events for 30 days would be wasteful.

In [ ]:
query = {
    'size': 0,
    'query': {
        'bool': {
            'filter': [
                {'range': {'@timestamp': {'gte': f'now-{LOOKBACK}'}}},
                {'term':  {'event.type': 'start'}}
            ]
        }
    },
    'aggs': {
        'parents': {
            'terms': {'field': 'process.parent.name', 'size': 200},
            'aggs': {
                'children': {'terms': {'field': 'process.name', 'size': 200}}
            }
        }
    }
}

resp = es.search(index=INDEX, body=query)

rows = []
for p in resp['aggregations']['parents']['buckets']:
    for c in p['children']['buckets']:
        rows.append({'parent': p['key'], 'child': c['key'], 'count': c['doc_count']})

df = pd.DataFrame(rows)
df['pct'] = df['count'] / df['count'].sum() * 100
df = df.sort_values('count', ascending=False).reset_index(drop=True)
df.head(20)

## 3. Surface rare pairs

Pairs seen less than 0.01% of the time are candidates for hunt review. Tune the threshold to your estate size.

In [ ]:
RARE_PCT = 0.01
rare = df[df['pct'] < RARE_PCT].copy()
print(f'{len(rare)} rare pairs found over the last {LOOKBACK}.')
rare.head(30)

## 4. Pivot into raw events

Pick a parent/child pair from the table above and pull the underlying events to triage.

In [ ]:
PARENT = 'explorer.exe'   # change me
CHILD  = 'mshta.exe'      # change me

pivot = {
    'size': 50,
    'query': {
        'bool': {
            'filter': [
                {'range': {'@timestamp': {'gte': f'now-{LOOKBACK}'}}},
                {'term':  {'event.type': 'start'}},
                {'term':  {'process.parent.name': PARENT}},
                {'term':  {'process.name':        CHILD}}
            ]
        }
    },
    'sort': [{'@timestamp': 'desc'}],
    '_source': ['@timestamp', 'host.name', 'user.name',
                'process.command_line', 'process.parent.command_line']
}

events = es.search(index=INDEX, body=pivot)
events_df = pd.DataFrame([h['_source'] for h in events['hits']['hits']])
events_df

## 5. Next steps

- If a pair looks suspicious, capture it as a hypothesis under the matching tactic folder.
- Promote anything reproducible to `rules/elastic/`.
- Re-run this notebook monthly and diff against the previous baseline.